### Research recap — parameters and provenance

- **From [`constants.py`](constants.py):** `UNB_MAG`, `UNB_PHASE`, node indices (`DISK_NODE`, etc.), `FREQ_RANGE` where used, and other shared symbols imported in the first code cell.
- **Notebook-local:** Modal speed `Q_(100, "rad/s")`, mode count, and plot-only choices.
- **Thesis role:** Validates undamped/damped natural frequencies against Sinha’s published benchmark before nonlinear fault models.



In [ ]:
import numpy as np
import ross as rs
import plotly.io as pio

from constants import *

pio.renderers.default = "notebook"
print(f"ROSS version: {rs.__version__}")

rotor = rs.Rotor.load("sinha_rotor.toml")

assert rotor.disk_elements[0].n == DISK_NODE, (
    f"sinha_rotor.toml disk node {rotor.disk_elements[0].n} != constants.DISK_NODE {DISK_NODE}"
)
assert rotor.bearing_elements[1].n == BEARING_2_NODE, (
    f"sinha_rotor.toml bearing-2 node {rotor.bearing_elements[1].n} != constants.BEARING_2_NODE {BEARING_2_NODE}"
)

In [ ]:
modal = rotor.run_modal(speed=Q_(100, "rad/s"), num_modes=24)

print("\nUndamped natural frequencies (Hz):")
print((modal.wn / (2 * np.pi)))

print("\nDamped natural frequencies (Hz):")
print((modal.wd / (2 * np.pi)))

print("\nDamping ratios:")
print(modal.damping_ratio)

In [ ]:
modal.plot_mode_3d(mode=1, frequency_units="Hz", animation=True)

## Mode shapes — all 6 modes (3 × 2 widescreen grid)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Configure which 6 modes to show in the 3 × 2 grid ───────────────────────
# Modes come in pairs (xy planes). Pick one from each pair to reduce clutter,
# or show consecutive modes to compare planes side-by-side.
#   first six         : modes = list(range(6))
#   every other (even): modes = list(range(0, 12, 2))   # [0,2,4,6,8,10]
#   every other (odd) : modes = list(range(1, 12, 2))   # [1,3,5,7,9,11]
modes = list(range(0, 12, 2))  # ← change this line to select a different set
# ─────────────────────────────────────────────────────────────────────────────

assert len(modes) == 6, "Grid is fixed at 3 × 2; supply exactly 6 mode indices."

wn_hz = modal.wn / (2 * np.pi)

sub_figs = [
    modal.plot_mode_3d(mode=i, frequency_units="Hz", animation=True)
    for i in modes
]

trace_offsets = []
running = 0
for sub_fig in sub_figs:
    trace_offsets.append(running)
    running += len(sub_fig.data)

subplot_titles = [f"Mode {i + 1}  ·  {wn_hz[i]:.2f} Hz" for i in modes]

fig = make_subplots(
    rows=2,
    cols=3,
    specs=[[{"type": "scene"}] * 3] * 2,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.02,
    vertical_spacing=0.08,
)

scene_slots = ["scene", "scene2", "scene3", "scene4", "scene5", "scene6"]

for idx, sub_fig in enumerate(sub_figs):
    row = idx // 3 + 1
    col = idx % 3 + 1
    for trace in sub_fig.data:
        fig.add_trace(trace, row=row, col=col)
    scene_json = sub_fig.layout.scene.to_plotly_json()
    if scene_json:
        fig.layout[scene_slots[idx]].update(scene_json)

combined_frames = []
for f_idx in range(len(sub_figs[0].frames)):
    data, traces = [], []
    for sub_idx, sub_fig in enumerate(sub_figs):
        base = trace_offsets[sub_idx]
        for t_idx, trace in enumerate(sub_fig.frames[f_idx].data):
            data.append(trace)
            traces.append(base + t_idx)
    combined_frames.append(go.Frame(data=data, traces=traces))
fig.frames = combined_frames

fig.update_layout(
    height=700,
    width=1400,
    title_text="Sinha rotor — undamped mode shapes",
    title_x=0.5,
    showlegend=False,
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            y=1.08,
            x=0.5,
            xanchor="center",
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[None, {"frame": {"duration": 50, "redraw": True}, "fromcurrent": True}],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                ),
            ],
        )
    ],
)

# fig.show()
# Opens the same figure in a new browser tab — press F11 for fullscreen
fig.show(renderer="browser")

## Unbalance response

In [ ]:
nodes = [DISK_NODE]
unbalance_magnitude = [UNB_MAG.to("kg*m").m]
unbalance_phase = [UNB_PHASE.to("rad").m]

speed = np.linspace(1e-3, 300, 1000)

resp = rotor.run_unbalance_response(
    node=nodes,
    unbalance_magnitude=unbalance_magnitude,
    unbalance_phase=unbalance_phase,
    frequency=speed,
)

## Deflected shapes for a healthy rotor

Here we plot the deflected shape for the speeds as reported by sinha.

In [ ]:
for spd in SPEEDS:
    target_rad_s = spd.to("rad/s").m
    idx = np.argmin(np.abs(resp.speed_range - target_rad_s))
    plot_rad_s = resp.speed_range[idx]

    fig = resp.plot_deflected_shape(speed=plot_rad_s)
    fig.update_layout(
        title=(
            f"Deflected shape near {spd.m:.0f} rpm "
            f"(plotted at {plot_rad_s:.2f} rad/s)"
        )
    )
    fig.show()

# Bode Plot

In [ ]:
probe = [rs.Probe(node=PROBE_NODE, angle=0.0)]
resp.plot_bode(probe, phase_units="deg", frequency_units="Hz")